# Recomendador de Carreras Universitarias del Ecuador
### Demo del backend para la asignatura *Programación para Inteligencia Artificial*

**Autor:** Arturo Rodríguez, PhD ([ORCID 0000-0002-7017-9443](https://orcid.org/0000-0002-7017-9443)) — docente ULEAM.

Este notebook ejecuta, en un solo archivo de Colab, una versión didáctica del backend real
del proyecto **Recomendador de Carreras Ecuador**: un sistema de orientación vocacional que
cruza un test de intereses (modelo **RIASEC** de Holland) con la oferta académica de las
Instituciones de Educación Superior (IES) del Ecuador, usando **scikit-learn**.

**Objetivo del notebook:** explicar, paso a paso, cómo funciona cada algoritmo del motor de
recomendación (filtro, TF-IDF, similitud coseno, `NearestNeighbors`, distancia de Haversine,
`KMeans`) y dejarte probar el motor completo con tu propio perfil vocacional.

> ⚠️ **Importante:** este notebook usa una **muestra reducida de datos**, escrita a mano más
> abajo, con fines exclusivamente didácticos. El sistema real corre sobre **8014 carreras**
> ofertadas por las IES del Ecuador y **99 cantones** con coordenadas geográficas (fuente:
> SENESCYT, Portal Único de Datos Abiertos del Ecuador, 5 de febrero de 2025). No se sube
> ningún archivo de datos a Colab: todo el código de esta celda en adelante es autocontenido.

**Repositorio y app real desplegada:** ver la última sección de este notebook.


## 1. Arquitectura del sistema real

El proyecto completo tiene 4 capas. Este notebook reproduce las capas 2 y 3 (motor de
recomendación) sobre datos de muestra; las capas 1 y 4 se describen pero no se ejecutan acá.

```
[1] Pipeline de datos (offline, se corre una sola vez)
    Excel SENESCYT (oferta académica cruda)
        -> src/01_limpiar_oferta.py        (limpieza, normalización de tildes/campos)
        -> src/02_cantones_coordenadas.py  (cruza 99 cantones con lat/lon)
        -> src/03_mapeo_riasec.py          (tabla curada: campo amplio -> pesos RIASEC)
        -> data/processed/*.csv            (oferta_limpia, cantones_coordenadas,
                                             mapeo_riasec_campo_amplio)

[2] Motor de recomendación (src/04_motor_recomendacion.py)
    Filtro duro (pandas) + vector RIASEC por carrera (campo amplio + TF-IDF)
    + búsqueda por similitud (NearestNeighbors, coseno) + cercanía (Haversine)
    + exploración por clústeres (KMeans)
    -> ESTE NOTEBOOK reconstruye esta capa completa, celda por celda, más abajo.

[3] API REST (backend/main.py, FastAPI) -- envuelve el motor en 6 endpoints:
    GET  /api/salud              -- healthcheck
    GET  /api/test-riasec        -- devuelve los 60 ítems del test + info de dimensiones
    POST /api/calcular-perfil    -- respuestas del test -> puntaje RIASEC 0-1 por dimensión
    GET  /api/opciones           -- valores disponibles para los filtros del frontend
    POST /api/recomendar         -- perfil + preferencias -> TODAS las carreras que pasan
                                     el filtro duro, ordenadas por score_final
    POST /api/comentario-perfil  -- comentario opcional generado por IA (Groq) sobre el
                                     perfil vocacional (ver nota más abajo)

[4] Frontend (frontend/, vanilla JS) -- 3 pantallas: test -> perfil/preferencias -> resultados
    (lista de carreras + "Mapa de afinidad", un diagrama de círculos concéntricos)
```

Cada sección de este notebook está rotulada con el endpoint o archivo real que reproduce,
para que puedas ubicarte en el código fuente del proyecto si querés profundizar.


## 2. Setup

Instalamos (si hiciera falta) y cargamos las librerías. Todas son las mismas que usa el
backend real (ver `requirements.txt`): `pandas`, `numpy`, `scikit-learn`. Agregamos
`matplotlib` solo para un gráfico ilustrativo de `KMeans` que no existe en el backend real
(el backend no genera gráficos, solo devuelve JSON).


In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

pd.set_option("display.max_colwidth", 60)
print("Librerías cargadas.")


## 3. Datos de muestra

En producción, el motor lee tres CSV desde `data/processed/`:

| Archivo real | Filas reales | Contenido |
|---|---|---|
| `oferta_limpia.csv` | 8014 | una fila por (carrera, IES, modalidad) vigente |
| `mapeo_riasec_campo_amplio.csv` | 10 | pesos RIASEC por campo amplio (curado a mano) |
| `cantones_coordenadas.csv` | 99 | lat/lon por cantón del Ecuador |

Acá, en vez de leer esos archivos, los reemplazamos por **subconjuntos reales** escritos a
mano en las celdas de abajo: mismas columnas, mismos nombres de carrera/IES/campo amplio
reales, pero solo ~25 filas (oferta), 10 filas (mapeo RIASEC — la tabla completa, es chica)
y 15 cantones. Así el notebook no depende de subir ningún archivo a Colab.


In [ ]:
# --- Oferta académica de muestra (mismas columnas que data/processed/oferta_limpia.csv) ---
oferta_muestra = pd.DataFrame([
    # NOMBRE_IES, TIPO_IES, TIPO_FINANCIAMIENTO, NOMBRE_CARRERA, CAMPO_AMPLIO_NORMALIZADO,
    # NIVEL_FORMACIÓN, MODALIDAD, PROVINCIA, CANTÓN, ES_PREGRADO, CANTON_KEY, PROVINCIA_KEY
    ["UNIVERSIDAD CENTRAL DEL ECUADOR", "UNIVERSIDAD", "PÚBLICA", "PSICOLOGIA CLINICA",
     "Ciencias Sociales, Periodismo e Información", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "PICHINCHA", "QUITO", True, "QUITO", "PICHINCHA"],
    ["UNIVERSIDAD CENTRAL DEL ECUADOR", "UNIVERSIDAD", "PÚBLICA", "TRABAJO SOCIAL",
     "Ciencias Sociales, Periodismo e Información", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "PICHINCHA", "QUITO", True, "QUITO", "PICHINCHA"],
    ["PONTIFICIA UNIVERSIDAD CATOLICA DEL ECUADOR", "UNIVERSIDAD", "PARTICULAR COFINANCIADA",
     "DERECHO", "Administración de Empresas y Derecho", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "PICHINCHA", "QUITO", True, "QUITO", "PICHINCHA"],
    ["UNIVERSIDAD DE LAS FUERZAS ARMADAS ESPE", "UNIVERSIDAD", "PÚBLICA",
     "ADMINISTRACION DE EMPRESAS", "Administración de Empresas y Derecho",
     "TERCER NIVEL O PREGRADO", "PRESENCIAL", "PICHINCHA", "QUITO", True, "QUITO", "PICHINCHA"],
    ["ESCUELA POLITECNICA NACIONAL", "UNIVERSIDAD", "PÚBLICA", "INGENIERIA EN SISTEMAS",
     "Tecnologías de la Información y la Comunicación (TIC)", "TERCER NIVEL O PREGRADO",
     "PRESENCIAL", "PICHINCHA", "QUITO", True, "QUITO", "PICHINCHA"],
    ["ESCUELA POLITECNICA NACIONAL", "UNIVERSIDAD", "PÚBLICA", "INGENIERIA EN SOFTWARE",
     "Tecnologías de la Información y la Comunicación (TIC)", "TERCER NIVEL O PREGRADO",
     "PRESENCIAL", "PICHINCHA", "QUITO", True, "QUITO", "PICHINCHA"],
    ["ESCUELA SUPERIOR POLITECNICA DEL LITORAL", "UNIVERSIDAD", "PÚBLICA",
     "INGENIERIA MECANICA", "Ingeniería, Industria y Construcción", "TERCER NIVEL O PREGRADO",
     "PRESENCIAL", "GUAYAS", "GUAYAQUIL", True, "GUAYAQUIL", "GUAYAS"],
    ["ESCUELA SUPERIOR POLITECNICA DEL LITORAL", "UNIVERSIDAD", "PÚBLICA",
     "INGENIERIA CIVIL", "Ingeniería, Industria y Construcción", "TERCER NIVEL O PREGRADO",
     "PRESENCIAL", "GUAYAS", "GUAYAQUIL", True, "GUAYAQUIL", "GUAYAS"],
    ["UNIVERSIDAD DE GUAYAQUIL", "UNIVERSIDAD", "PÚBLICA", "MEDICINA",
     "Salud y Bienestar", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "GUAYAS", "GUAYAQUIL", True, "GUAYAQUIL", "GUAYAS"],
    ["UNIVERSIDAD DE GUAYAQUIL", "UNIVERSIDAD", "PÚBLICA", "ENFERMERIA",
     "Salud y Bienestar", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "GUAYAS", "GUAYAQUIL", True, "GUAYAQUIL", "GUAYAS"],
    ["UNIVERSIDAD DE CUENCA", "UNIVERSIDAD", "PÚBLICA", "ARQUITECTURA",
     "Ingeniería, Industria y Construcción", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "AZUAY", "CUENCA", True, "CUENCA", "AZUAY"],
    ["UNIVERSIDAD DE CUENCA", "UNIVERSIDAD", "PÚBLICA", "BIOLOGIA",
     "Ciencias Naturales, Matemáticas y Estadística", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "AZUAY", "CUENCA", True, "CUENCA", "AZUAY"],
    ["UNIVERSIDAD LAICA ELOY ALFARO DE MANABI", "UNIVERSIDAD", "PÚBLICA",
     "PEDAGOGIA DE LA ACTIVIDAD FISICA Y DEPORTE", "Educación", "TERCER NIVEL O PREGRADO",
     "PRESENCIAL", "MANABI", "MANTA", True, "MANTA", "MANABI"],
    ["UNIVERSIDAD LAICA ELOY ALFARO DE MANABI", "UNIVERSIDAD", "PÚBLICA",
     "EDUCACION BASICA", "Educación", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "MANABI", "MANTA", True, "MANTA", "MANABI"],
    ["UNIVERSIDAD LAICA ELOY ALFARO DE MANABI", "UNIVERSIDAD", "PÚBLICA",
     "INGENIERIA EN ACUICULTURA", "Agricultura, Silvicultura, Pesca y Veterinaria",
     "TERCER NIVEL O PREGRADO", "PRESENCIAL", "MANABI", "MANTA", True, "MANTA", "MANABI"],
    ["UNIVERSIDAD TECNICA DE MANABI", "UNIVERSIDAD", "PÚBLICA", "MEDICINA VETERINARIA",
     "Agricultura, Silvicultura, Pesca y Veterinaria", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "MANABI", "PORTOVIEJO", True, "PORTOVIEJO", "MANABI"],
    ["UNIVERSIDAD TECNICA DE AMBATO", "UNIVERSIDAD", "PÚBLICA", "DISEÑO GRAFICO",
     "Artes y Humanidades", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "TUNGURAHUA", "AMBATO", True, "AMBATO", "TUNGURAHUA"],
    ["UNIVERSIDAD TECNICA DE AMBATO", "UNIVERSIDAD", "PÚBLICA", "GASTRONOMIA",
     "Servicios", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "TUNGURAHUA", "AMBATO", True, "AMBATO", "TUNGURAHUA"],
    ["UNIVERSIDAD TECNICA PARTICULAR DE LOJA", "UNIVERSIDAD", "PARTICULAR AUTOFINANCIADA",
     "CONTABILIDAD Y AUDITORIA", "Administración de Empresas y Derecho",
     "TERCER NIVEL O PREGRADO", "A DISTANCIA", "LOJA", "LOJA", True, "LOJA", "LOJA"],
    ["UNIVERSIDAD TECNICA PARTICULAR DE LOJA", "UNIVERSIDAD", "PARTICULAR AUTOFINANCIADA",
     "SECRETARIADO EJECUTIVO", "Administración de Empresas y Derecho",
     "TERCER NIVEL O PREGRADO", "A DISTANCIA", "LOJA", "LOJA", True, "LOJA", "LOJA"],
    ["INSTITUTO SUPERIOR TECNOLOGICO ISMAC", "INSTITUTO", "PARTICULAR AUTOFINANCIADA",
     "TECNOLOGIA EN MECANICA AUTOMOTRIZ", "Ingeniería, Industria y Construcción",
     "TÉCNICO/TECNOLÓGICO SUPERIOR", "PRESENCIAL", "PICHINCHA", "QUITO", True, "QUITO",
     "PICHINCHA"],
    ["UNIVERSIDAD TECNICA DEL NORTE", "UNIVERSIDAD", "PÚBLICA", "AGROINDUSTRIA",
     "Agricultura, Silvicultura, Pesca y Veterinaria", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "IMBABURA", "IBARRA", True, "IBARRA", "IMBABURA"],
    ["UNIVERSIDAD TECNICA DEL NORTE", "UNIVERSIDAD", "PÚBLICA", "ESTADISTICA",
     "Ciencias Naturales, Matemáticas y Estadística", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "IMBABURA", "IBARRA", True, "IBARRA", "IMBABURA"],
    ["ESCUELA SUPERIOR POLITECNICA DE CHIMBORAZO", "UNIVERSIDAD", "PÚBLICA",
     "INGENIERIA AMBIENTAL", "Ciencias Naturales, Matemáticas y Estadística",
     "TERCER NIVEL O PREGRADO", "PRESENCIAL", "CHIMBORAZO", "RIOBAMBA", True, "RIOBAMBA",
     "CHIMBORAZO"],
    ["UNIVERSIDAD ESTATAL DE MILAGRO", "UNIVERSIDAD", "PÚBLICA", "MARKETING",
     "Administración de Empresas y Derecho", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "GUAYAS", "MILAGRO", True, "MILAGRO", "GUAYAS"],
    ["UNIVERSIDAD DE LAS ARTES", "UNIVERSIDAD", "PÚBLICA", "ARTES MUSICALES",
     "Artes y Humanidades", "TERCER NIVEL O PREGRADO", "PRESENCIAL",
     "GUAYAS", "GUAYAQUIL", True, "GUAYAQUIL", "GUAYAS"],
], columns=[
    "NOMBRE_IES", "TIPO_IES", "TIPO_FINANCIAMIENTO", "NOMBRE_CARRERA",
    "CAMPO_AMPLIO_NORMALIZADO", "NIVEL_FORMACIÓN", "MODALIDAD", "PROVINCIA", "CANTÓN",
    "ES_PREGRADO", "CANTON_KEY", "PROVINCIA_KEY",
])

print(f"Oferta de muestra: {len(oferta_muestra)} filas "
      f"(la real, oferta_limpia.csv, tiene 8014). "
      f"Campos amplios representados: {oferta_muestra['CAMPO_AMPLIO_NORMALIZADO'].nunique()} de 10.")
oferta_muestra.head()


In [ ]:
# --- Mapeo RIASEC <-> campo amplio (tabla completa real, 10 filas -- ver src/03_mapeo_riasec.py) ---
mapeo_riasec = pd.DataFrame([
    ["Educación",                                                    0.0, 0.1, 0.2, 1.0, 0.1, 0.1],
    ["Artes y Humanidades",                                          0.0, 0.1, 1.0, 0.3, 0.1, 0.0],
    ["Ciencias Sociales, Periodismo e Información",                  0.0, 0.2, 0.3, 0.4, 0.6, 0.1],
    ["Administración de Empresas y Derecho",                         0.0, 0.1, 0.0, 0.2, 0.7, 0.5],
    ["Ciencias Naturales, Matemáticas y Estadística",                0.2, 1.0, 0.0, 0.0, 0.0, 0.2],
    ["Tecnologías de la Información y la Comunicación (TIC)",        0.3, 0.4, 0.1, 0.0, 0.2, 0.6],
    ["Ingeniería, Industria y Construcción",                         0.8, 0.3, 0.1, 0.0, 0.1, 0.1],
    ["Agricultura, Silvicultura, Pesca y Veterinaria",                0.9, 0.3, 0.0, 0.1, 0.0, 0.0],
    ["Salud y Bienestar",                                            0.1, 0.6, 0.0, 0.6, 0.1, 0.1],
    ["Servicios",                                                    0.5, 0.1, 0.1, 0.5, 0.3, 0.3],
], columns=["campo_amplio_normalizado", "R", "I", "A", "S", "E", "C"])

mapeo_riasec


In [ ]:
# --- Cantones con coordenadas (subconjunto real de cantones_coordenadas.csv, 99 filas en total) ---
cantones_muestra = pd.DataFrame([
    ["PICHINCHA", "QUITO", -0.1806532, -78.4678382],
    ["GUAYAS", "GUAYAQUIL", -2.1709979, -79.9224075],
    ["AZUAY", "CUENCA", -2.8973745, -79.0044518],
    ["TUNGURAHUA", "AMBATO", -1.2417721, -78.6197361],
    ["MANABI", "MANTA", -0.9676533, -80.7089273],
    ["MANABI", "PORTOVIEJO", -1.0544940, -80.4560423],
    ["LOJA", "LOJA", -3.9930900, -79.2041780],
    ["IMBABURA", "IBARRA", 0.3391781, -78.1224649],
    ["CHIMBORAZO", "RIOBAMBA", -1.6635508, -78.6547632],
    ["GUAYAS", "MILAGRO", -2.1345655, -79.5945423],
    ["EL ORO", "MACHALA", -3.2581112, -79.9553924],
    ["SANTO DOMINGO DE LOS TSACHILAS", "SANTO DOMINGO", -0.2531668, -79.1746209],
    ["CARCHI", "TULCAN", 0.8113169, -77.7172942],
    ["COTOPAXI", "LATACUNGA", -0.9345312, -78.6157569],
    ["LOS RIOS", "BABAHOYO", -1.8022241, -79.5346591],
], columns=["provincia", "canton", "lat", "lon"])
cantones_muestra["canton_key"] = cantones_muestra["canton"]
cantones_muestra["provincia_key"] = cantones_muestra["provincia"]

print(f"Cantones de muestra: {len(cantones_muestra)} (la real, cantones_coordenadas.csv, tiene 99).")
cantones_muestra


## 4. El cuestionario vocacional RIASEC

Endpoint real: `GET /api/test-riasec` + `POST /api/calcular-perfil`
(archivo: `backend/test_riasec.py`).

El test se basa en el modelo **RIASEC de Holland**, instrumentado con el
**O*NET Interest Profiler Short Form** (National Center for O*NET Development, 2010, US
Dept. of Labor) — un instrumento de licencia abierta, traducido al español para este
proyecto. Consta de **60 actividades** (10 por cada una de las 6 dimensiones), presentadas
en orden intercalado para no revelarle al estudiante qué dimensión mide cada una:

| Dimensión | Nombre | Descripción |
|---|---|---|
| **R** | Realista | Te gusta trabajar con las manos, herramientas, máquinas o al aire libre. |
| **I** | Investigativo | Te gusta observar, investigar, analizar y resolver problemas complejos. |
| **A** | Artístico | Te gusta crear, expresarte artísticamente y trabajar sin reglas fijas. |
| **S** | Social | Te gusta ayudar, enseñar, cuidar o trabajar directamente con personas. |
| **E** | Emprendedor | Te gusta liderar, persuadir, emprender y tomar decisiones de negocio. |
| **C** | Convencional | Te gusta organizar datos, seguir procedimientos claros y trabajar con precisión. |

El estudiante responde "sí" o "no" a cada actividad. El puntaje de cada dimensión es
simplemente:

$$\text{puntaje}_d = \frac{\text{cantidad de "sí" en los 10 ítems de la dimensión } d}{10}$$

...un número entre 0 y 1 por cada una de las 6 dimensiones. Ese vector de 6 números es el
**perfil vocacional** que después el motor compara contra cada carrera.

Algunos ítems reales de ejemplo (2 por dimensión, de los 60 totales):

| Dimensión | Ítem de ejemplo |
|---|---|
| R | "Reparar electrodomésticos" |
| R | "Ensamblar partes electrónicas" |
| I | "Realizar experimentos químicos" |
| I | "Trabajar en un laboratorio de biología" |
| A | "Tocar un instrumento musical" |
| A | "Crear efectos especiales para películas" |
| S | "Dar orientación vocacional o de carrera a otras personas" |
| S | "Dar clases en un colegio" |
| E | "Iniciar tu propio negocio" |
| E | "Negociar contratos comerciales" |
| C | "Llevar el registro del inventario" |
| C | "Corregir y revisar registros o formularios" |

**Este notebook no reimplementa el formulario de 60 ítems** (respondería lento en una
presentación). En su lugar, en la siguiente celda vos ponés directamente el resultado del
cálculo — el vector de 6 números — como si ya hubieras respondido el test.


## 5. Tu perfil vocacional

Editá los 6 valores de abajo (entre 0 y 1) según tus propios intereses y volvé a correr
esta celda y las que siguen. No hace falta que sumen 1 -- el motor los normaliza. Guía de
referencia (mientras más alto, más te interesa esa dimensión):

- **R** (Realista): trabajo manual, técnico, mecánico, al aire libre.
- **I** (Investigativo): investigación, ciencia, análisis, laboratorio.
- **A** (Artístico): arte, diseño, música, creatividad.
- **S** (Social): ayudar, enseñar, cuidar, trabajar con personas.
- **E** (Emprendedor): liderazgo, negocios, ventas, persuasión.
- **C** (Convencional): organización, datos, procedimientos, precisión.


In [ ]:
# Editá estos valores (0.0 a 1.0) y volvé a correr desde acá hacia abajo
perfil_riasec = {
    "R": 0.2,   # Realista
    "I": 0.6,   # Investigativo
    "A": 0.1,   # Artístico
    "S": 0.8,   # Social
    "E": 0.3,   # Emprendedor
    "C": 0.2,   # Convencional
}
perfil_riasec


## 6. Algoritmo 1 — Filtro duro (pandas)

Archivo real: `MotorRecomendacion._filtrar_duro()` en `src/04_motor_recomendacion.py`.

Antes de calcular ninguna similitud, se descartan con **filtros booleanos de pandas** las
ofertas que no cumplen una preferencia obligatoria del estudiante: modalidad, tipo de
financiamiento, tipo de IES, nivel de formación (por defecto excluye posgrado). Esto no es
machine learning -- es álgebra de conjuntos sobre un `DataFrame`, pero es el primer paso
necesario: reduce el espacio de búsqueda antes de que el resto de los algoritmos trabajen.


In [ ]:
# Demo aislada: filtrar por modalidad y financiamiento
def filtrar_duro(df, modalidad=None, financiamiento=None):
    resultado = df[df["ES_PREGRADO"]]
    if modalidad:
        resultado = resultado[resultado["MODALIDAD"].str.upper() == modalidad.upper()]
    if financiamiento:
        resultado = resultado[resultado["TIPO_FINANCIAMIENTO"].str.upper() == financiamiento.upper()]
    return resultado

antes = len(oferta_muestra)
filtrado_demo = filtrar_duro(oferta_muestra, modalidad="PRESENCIAL", financiamiento="PÚBLICA")
print(f"Antes del filtro: {antes} filas -> después (PRESENCIAL + PÚBLICA): {len(filtrado_demo)} filas")
filtrado_demo[["NOMBRE_IES", "NOMBRE_CARRERA", "MODALIDAD", "TIPO_FINANCIAMIENTO"]]


## 7. Algoritmo 2 — Vector RIASEC por carrera (campo amplio + TF-IDF)

Archivo real: `MotorRecomendacion.__init__()` y `_vector_texto_por_carrera()` en
`src/04_motor_recomendacion.py`.

Cada carrera necesita un **vector de 6 dimensiones (R,I,A,S,E,C)** para poder compararse
contra el perfil del estudiante. La fuente principal es su `CAMPO_AMPLIO_NORMALIZADO` (10
categorías, tabla de la sección 3) -- pero eso solo da **10 vectores distintos posibles**
para 8014 carreras. En la práctica, esto hacía que, por ejemplo, las **338 carreras
distintas** clasificadas como "Administración de Empresas y Derecho" quedaran **todas
empatadas en 99.5% de afinidad** con cualquier estudiante -- sin diferenciarse entre sí.

**Solución:** mezclar el vector de campo amplio con una señal de texto sacada del propio
`NOMBRE_CARRERA`, usando **TF-IDF** (Term Frequency - Inverse Document Frequency) +
**similitud coseno** contra una lista de palabras clave curada por dimensión
(`PALABRAS_CLAVE_DIMENSION`, teoría de Holland). La mezcla final es
**85% campo amplio + 15% señal de texto** -- el campo amplio sigue mandando, el texto solo
desempata carreras que hoy comparten vector exacto.


In [ ]:
# Palabras clave por dimensión RIASEC (copiadas tal cual de src/04_motor_recomendacion.py)
PALABRAS_CLAVE_DIMENSION = {
    "R": "mecanica mecanico electricidad electronica construccion agropecuaria "
         "agricola veterinaria minas industrial automotriz mantenimiento tecnico "
         "obras civil forestal pesca manufactura maquinaria",
    "I": "investigacion ciencia cientifico analisis biologia quimica fisica "
         "matematica estadistica laboratorio biotecnologia ambiental geologia",
    "A": "arte artistico diseno musica teatro danza cine fotografia moda "
         "creativo literatura escritura audiovisual publicidad",
    "S": "social educacion docencia psicologia enfermeria salud terapia "
         "comunitario cuidado orientacion ensenanza pedagogia",
    "E": "gestion negocio empresa empresarial liderazgo marketing ventas "
         "comercio emprendimiento finanzas administracion gerencia negociacion",
    "C": "contabilidad auditoria control administrativo secretariado archivo "
         "datos tributacion logistica procesos calidad",
}
DIMENSIONES = ["R", "I", "A", "S", "E", "C"]

# --- Demo aislada de TF-IDF: 4 nombres de carrera vs. las 6 "anclas" de dimensión ---
nombres_demo = ["DERECHO", "ADMINISTRACION DE EMPRESAS", "MARKETING", "CONTABILIDAD Y AUDITORIA"]
anclas = [PALABRAS_CLAVE_DIMENSION[d] for d in DIMENSIONES]
corpus_demo = [n.lower() for n in nombres_demo] + anclas

vectorizador_demo = TfidfVectorizer(strip_accents="unicode")
matriz_demo = vectorizador_demo.fit_transform(corpus_demo)
sim_demo = cosine_similarity(matriz_demo[:len(nombres_demo)], matriz_demo[len(nombres_demo):])

tabla_demo = pd.DataFrame(sim_demo, index=nombres_demo, columns=DIMENSIONES).round(3)
print("Similitud coseno TF-IDF de cada nombre de carrera contra cada dimensión RIASEC:")
print("(antes del TF-IDF, las 4 compartían EXACTAMENTE el mismo vector: el de su campo amplio)")
tabla_demo


Observá que **"CONTABILIDAD Y AUDITORIA"** saca más señal en **C** (Convencional) y
**"MARKETING"** más en **E** (Emprendedor), aunque ambas pertenecen al mismo campo amplio
("Administración de Empresas y Derecho") y antes del TF-IDF tenían el vector RIASEC
idéntico. Esa es la diferenciación que resuelve el problema de las 338 carreras empatadas.

Ahora armamos el vector final de cada carrera de la muestra, con la mezcla 85%/15%:


In [ ]:
def vector_texto_por_carrera(df):
    nombres_unicos = df["NOMBRE_CARRERA"].drop_duplicates().reset_index(drop=True)
    anclas = [PALABRAS_CLAVE_DIMENSION[d] for d in DIMENSIONES]
    corpus = list(nombres_unicos.str.lower()) + anclas

    vectorizador = TfidfVectorizer(strip_accents="unicode")
    matriz = vectorizador.fit_transform(corpus)
    similitud = cosine_similarity(matriz[:len(nombres_unicos)], matriz[len(nombres_unicos):])
    similitud[similitud < 0] = 0.0

    tabla = pd.DataFrame(similitud, columns=DIMENSIONES)
    tabla["NOMBRE_CARRERA"] = nombres_unicos.values
    vec_texto_df = df[["NOMBRE_CARRERA"]].merge(tabla, on="NOMBRE_CARRERA", how="left")
    return vec_texto_df[DIMENSIONES].to_numpy(dtype=float)


def armar_vector_final(df, mapeo):
    df = df.merge(mapeo, left_on="CAMPO_AMPLIO_NORMALIZADO", right_on="campo_amplio_normalizado", how="left")

    pesos_campo = df[DIMENSIONES].to_numpy(dtype=float)
    sumas_campo = pesos_campo.sum(axis=1, keepdims=True)
    sumas_campo[sumas_campo == 0] = 1.0
    vec_campo = pesos_campo / sumas_campo

    vec_texto = vector_texto_por_carrera(df)
    sumas_texto = vec_texto.sum(axis=1, keepdims=True)
    tiene_texto = (sumas_texto.ravel() > 0)
    vec_texto_norm = np.zeros_like(vec_texto)
    vec_texto_norm[tiene_texto] = vec_texto[tiene_texto] / sumas_texto[tiene_texto]

    peso_texto = 0.15
    vec_final = vec_campo.copy()
    vec_final[tiene_texto] = (1 - peso_texto) * vec_campo[tiene_texto] + peso_texto * vec_texto_norm[tiene_texto]
    sumas_final = vec_final.sum(axis=1, keepdims=True)
    sumas_final[sumas_final == 0] = 1.0
    vec_final = vec_final / sumas_final

    df[[f"vec_{d}" for d in DIMENSIONES]] = vec_final
    return df

oferta_con_vector = armar_vector_final(oferta_muestra.copy(), mapeo_riasec)
oferta_con_vector[["NOMBRE_CARRERA", "CAMPO_AMPLIO_NORMALIZADO"] + [f"vec_{d}" for d in DIMENSIONES]].round(3)


## 8. Algoritmo 3 — Búsqueda por similitud (`sklearn.neighbors.NearestNeighbors`, métrica coseno)

Archivo real: `MotorRecomendacion.buscar()` en `src/04_motor_recomendacion.py`.

Con cada carrera ya representada como un punto en un espacio de 6 dimensiones (R,I,A,S,E,C),
el problema de "¿qué carreras se parecen a mi perfil?" se vuelve un problema clásico de
**búsqueda de vecinos más cercanos**. Se usa la **distancia coseno**, que mide el ángulo
entre dos vectores (no su magnitud) -- apropiado acá porque lo que importa es la *forma*
del perfil (qué dimensiones pesan más entre sí), no su escala:

$$\text{distancia\_coseno}(u, v) = 1 - \frac{u \cdot v}{\|u\| \, \|v\|} \qquad\Rightarrow\qquad \text{similitud} = 1 - \text{distancia}$$

**Detalle importante del motor real:** se le pide a `NearestNeighbors` el ranking
**completo** (`n_neighbors = k = todos los candidatos`), no un top-k chico. La razón: el
vector RIASEC de cada oferta viene mayormente de su campo amplio (10 categorías), así que
truncar temprano a un k pequeño puede dejar todo el resultado ocupado por un solo campo,
y ese pool completo ordenado es justamente lo que necesita el frontend para su slider de
"afinidad mínima" (ver sección 11).


In [ ]:
# Demo aislada: un vector de juguete contra 3 vectores de ejemplo
vectores_juguete = np.array([
    [0.8, 0.3, 0.1, 0.0, 0.1, 0.1],   # parecido a "Ingeniería, Industria y Construcción"
    [0.0, 0.1, 0.2, 1.0, 0.1, 0.1],   # parecido a "Educación"
    [0.0, 0.1, 0.0, 0.2, 0.7, 0.5],   # parecido a "Administración de Empresas y Derecho"
])
etiquetas_juguete = ["Perfil tipo Ingeniería", "Perfil tipo Educación", "Perfil tipo Administración"]

vector_consulta = np.array([[0.1, 0.2, 0.1, 0.9, 0.2, 0.1]])  # alto en S (Social)

vecinos_demo = NearestNeighbors(n_neighbors=3, metric="cosine")
vecinos_demo.fit(vectores_juguete)
distancias, indices = vecinos_demo.kneighbors(vector_consulta)

for rank, (idx, dist) in enumerate(zip(indices.ravel(), distancias.ravel()), start=1):
    print(f"{rank}. {etiquetas_juguete[idx]:30s}  similitud coseno = {1 - dist:.3f}")


Como se esperaba, el perfil "tipo Educación" (que también pesa alto en **S**) queda
primero. Ahora corremos lo mismo, pero contra los vectores reales de la oferta de muestra:


In [ ]:
def buscar_por_similitud(df_con_vector, perfil_dict):
    vector_estudiante = np.array([perfil_dict.get(d, 0.0) for d in DIMENSIONES], dtype=float)
    vector_estudiante = vector_estudiante / vector_estudiante.sum()

    matriz = df_con_vector[[f"vec_{d}" for d in DIMENSIONES]].to_numpy()
    k = len(df_con_vector)
    vecinos = NearestNeighbors(n_neighbors=k, metric="cosine")
    vecinos.fit(matriz)
    distancias, indices = vecinos.kneighbors(vector_estudiante.reshape(1, -1))

    resultado = df_con_vector.iloc[indices.ravel()].copy()
    resultado["similitud_riasec"] = 1 - distancias.ravel()
    return resultado.sort_values("similitud_riasec", ascending=False)

ranking_demo = buscar_por_similitud(oferta_con_vector, perfil_riasec)
ranking_demo[["NOMBRE_CARRERA", "CAMPO_AMPLIO_NORMALIZADO", "similitud_riasec"]].head(10).round(3)


## 9. Algoritmo 4 — Cercanía geográfica (fórmula de Haversine)

Archivo real: función `haversine_km()` en `src/04_motor_recomendacion.py`.

Para el filtro de "cercanía" (cuánto le importa al estudiante vivir cerca de la sede),
el motor calcula la distancia en línea recta entre el cantón del estudiante y el cantón de
cada oferta, usando la **fórmula de Haversine** -- distancia entre dos puntos sobre una
esfera a partir de su latitud/longitud:

$$a = \sin^2\!\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\!\left(\frac{\Delta\lambda}{2}\right)$$
$$d = 2r \cdot \arcsin(\sqrt{a})$$

donde $\phi$ es latitud, $\lambda$ es longitud (en radianes) y $r$ = 6371 km (radio de la
Tierra). Esta distancia después se invierte y se escala 0-1 (`MinMaxScaler`) para convertirla
en un *score* de cercanía combinable con la similitud RIASEC.


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

# Demo aislada: distancia entre Manta y el resto de cantones de muestra
manta = cantones_muestra[cantones_muestra["canton"] == "MANTA"].iloc[0]
distancias_demo = cantones_muestra.assign(
    distancia_km=lambda d: d.apply(lambda r: haversine_km(manta["lat"], manta["lon"], r["lat"], r["lon"]), axis=1)
).sort_values("distancia_km")

distancias_demo[["provincia", "canton", "distancia_km"]].round(1)


## 10. Algoritmo 5 — Exploración por clústeres (`sklearn.cluster.KMeans`)

Archivo real: `MotorRecomendacion.explorar_clusters_vocacionales()` en
`src/04_motor_recomendacion.py`.

Este algoritmo **no forma parte de una búsqueda puntual** (no responde "¿qué carrera me
conviene?") -- agrupa **todas** las carreras únicas del dataset en clústeres vocacionales,
pensado para una futura vista de "explorá por familia de interés" en el frontend. Usa
`KMeans` sobre el mismo espacio de 6 dimensiones (R,I,A,S,E,C) que ya construimos en la
sección 7.


In [ ]:
base_clusters = oferta_con_vector.drop_duplicates(subset=["NOMBRE_CARRERA", "CAMPO_AMPLIO_NORMALIZADO"]).copy()
matriz_clusters = base_clusters[[f"vec_{d}" for d in DIMENSIONES]].to_numpy()

n_clusters = min(4, len(base_clusters))
km = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
base_clusters["cluster_vocacional"] = km.fit_predict(matriz_clusters)

base_clusters[["NOMBRE_CARRERA", "CAMPO_AMPLIO_NORMALIZADO", "cluster_vocacional"]]


In [ ]:
# Visualización: reducimos las 6 dimensiones a 2 con PCA solo para poder graficar
coordenadas_2d = PCA(n_components=2, random_state=42).fit_transform(matriz_clusters)

plt.figure(figsize=(7, 5))
dispersión = plt.scatter(coordenadas_2d[:, 0], coordenadas_2d[:, 1],
                          c=base_clusters["cluster_vocacional"], cmap="tab10", s=80)
for (x, y), nombre in zip(coordenadas_2d, base_clusters["NOMBRE_CARRERA"]):
    plt.annotate(nombre[:18], (x, y), fontsize=7, alpha=0.75)
plt.title(f"Clústeres vocacionales (KMeans, k={n_clusters}) -- proyección PCA 2D")
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.tight_layout()
plt.show()


## 11. Motor integrado

Ahora juntamos los 5 algoritmos anteriores en una sola clase, replicando 1:1 la lógica de
`MotorRecomendacion.buscar()` (src/04_motor_recomendacion.py), pero operando sobre los
`DataFrame` de muestra en vez de los CSV reales. El orden es siempre el mismo:

1. Filtro duro (sección 6)
2. Vector RIASEC por carrera: campo amplio + TF-IDF (sección 7)
3. Búsqueda por similitud con `NearestNeighbors` (sección 8)
4. Score de cercanía con Haversine, si el estudiante le dio peso (sección 9)
5. `score_final = (1 - peso_cercania) * similitud_riasec + peso_cercania * score_cercania`
6. Deduplicar (misma carrera + misma IES) y ordenar de mayor a menor `score_final`

`KMeans` (sección 10) queda aparte a propósito: es exploratorio, no participa de una
búsqueda puntual.


In [ ]:
class MotorRecomendacionDemo:
    def __init__(self, oferta, cantones, mapeo):
        self.cantones = cantones
        self.oferta = armar_vector_final(
            oferta.merge(cantones[["provincia_key", "canton_key", "lat", "lon"]],
                         left_on=["PROVINCIA_KEY", "CANTON_KEY"],
                         right_on=["provincia_key", "canton_key"], how="left"),
            mapeo,
        )

    def _score_cercania(self, df, canton_estudiante):
        if not canton_estudiante:
            return pd.Series(0.0, index=df.index)
        ref = self.oferta.loc[self.oferta["CANTON_KEY"] == canton_estudiante.upper()]
        if ref.empty or pd.isna(ref.iloc[0]["lat"]):
            return pd.Series(0.0, index=df.index)
        lat0, lon0 = ref.iloc[0]["lat"], ref.iloc[0]["lon"]
        dist = df.apply(lambda r: haversine_km(lat0, lon0, r["lat"], r["lon"]), axis=1)
        dist = dist.fillna(dist.max() if dist.notna().any() else 0.0)
        if dist.max() == dist.min():
            return pd.Series(1.0, index=df.index)
        prox = 1 - MinMaxScaler().fit_transform(dist.to_numpy().reshape(-1, 1)).ravel()
        return pd.Series(prox, index=df.index)

    def buscar(self, perfil_riasec, modalidad=None, financiamiento=None,
               canton_estudiante=None, peso_cercania=0.0):
        vector_estudiante = np.array([perfil_riasec.get(d, 0.0) for d in DIMENSIONES], dtype=float)
        vector_estudiante = vector_estudiante / vector_estudiante.sum()

        candidatos = filtrar_duro(self.oferta, modalidad=modalidad, financiamiento=financiamiento)
        if candidatos.empty:
            return candidatos

        matriz = candidatos[[f"vec_{d}" for d in DIMENSIONES]].to_numpy()
        k = len(candidatos)
        vecinos = NearestNeighbors(n_neighbors=k, metric="cosine")
        vecinos.fit(matriz)
        distancias, indices = vecinos.kneighbors(vector_estudiante.reshape(1, -1))

        pool = candidatos.iloc[indices.ravel()].copy()
        pool["similitud_riasec"] = 1 - distancias.ravel()
        pool["score_cercania"] = self._score_cercania(pool, canton_estudiante).values
        pool["score_final"] = (1 - peso_cercania) * pool["similitud_riasec"] + peso_cercania * pool["score_cercania"]
        pool = pool.sort_values("score_final", ascending=False)

        resultado = pool.drop_duplicates(subset=["NOMBRE_CARRERA", "NOMBRE_IES"], keep="first")
        columnas = ["NOMBRE_CARRERA", "NOMBRE_IES", "CAMPO_AMPLIO_NORMALIZADO", "PROVINCIA", "CANTÓN",
                    "MODALIDAD", "TIPO_FINANCIAMIENTO", "similitud_riasec", "score_cercania", "score_final"]
        return resultado[columnas].reset_index(drop=True)


motor = MotorRecomendacionDemo(oferta_muestra.copy(), cantones_muestra, mapeo_riasec)
print("Motor listo. Corré la celda de la sección 12 con tu perfil de la sección 5.")


## 12. Probá el motor con tu perfil

Endpoint real equivalente: `POST /api/recomendar`.

Corré esta celda con el `perfil_riasec` que editaste en la sección 5. Si querés ver el
efecto de la cercanía geográfica, subí `peso_cercania` (0 = indiferente, 1 = solo importa
la cercanía) y elegí un `canton_estudiante` de la tabla de la sección 3 (ej. `"QUITO"`,
`"MANTA"`, `"CUENCA"`).

**Para experimentar:** volvé a la sección 5, cambiá `perfil_riasec`, y corré de nuevo esta
celda -- vas a ver que el orden de las carreras cambia según qué dimensión subiste.


In [ ]:
resultados = motor.buscar(
    perfil_riasec,
    modalidad=None,                 # ej. "PRESENCIAL" para filtrar
    financiamiento=None,            # ej. "PÚBLICA"
    canton_estudiante="QUITO",      # tu cantón, para el score de cercanía
    peso_cercania=0.2,              # 0 = indiferente a la cercanía ... 1 = solo importa eso
)

print(f"{len(resultados)} carreras de la muestra pasaron el filtro duro, ordenadas por afinidad:")
resultados[["NOMBRE_CARRERA", "NOMBRE_IES", "CAMPO_AMPLIO_NORMALIZADO",
            "CANTÓN", "similitud_riasec", "score_cercania", "score_final"]].round(3)


## 13. Nota: comentario del perfil generado por IA (no incluido en este notebook)

El backend real tiene un endpoint adicional, `POST /api/comentario-perfil`, que genera un
comentario breve (100-150 palabras) sobre el perfil vocacional del estudiante usando un
LLM (**Groq**, modelo `llama-3.1-8b-instant`). Recibe únicamente los 6 puntajes RIASEC y
los campos amplios ya calculados por el propio motor -- el prompt le prohíbe explícitamente
inventar carreras o universidades.

**Por qué no se replica acá:** ese endpoint necesita una `GROQ_API_KEY` guardada como
variable de entorno del lado del servidor (Render). Nunca debe vivir en un notebook público
ni en el frontend -- cualquier secreto ahí queda expuesto. Si falla o no está configurada,
el propio backend responde `503` y el frontend lo maneja en silencio (no bloquea el resto
de la app).


## 14. De la muestra al sistema real

| | Este notebook | Sistema real |
|---|---|---|
| Oferta académica | ~25 filas escritas a mano | **8014 filas** (`data/processed/oferta_limpia.csv`) |
| Cantones con coordenadas | 15 | **99** (`data/processed/cantones_coordenadas.csv`) |
| Campos amplios | 10 (tabla completa) | 10 (igual) |
| Fuente | subconjunto real, sin archivo | SENESCYT, Portal Único de Datos Abiertos del Ecuador (5-feb-2025) |

**Pendientes conocidos del proyecto** (ver `README.md`):

1. Aprendizaje supervisado (`RandomForestClassifier`) sobre feedback real de usuarios
   ("me interesó" / "no me interesó") para reentrenar el re-ranking.
2. Ampliar `PALABRAS_CLAVE_DIMENSION` con más vocabulario/sinónimos -- hoy son listas
   cortas curadas a mano.
3. Validar `mapeo_riasec_campo_amplio.csv` y `PALABRAS_CLAVE_DIMENSION` con un orientador
   vocacional o psicólogo educativo (ambos curados a mano por el autor, no por un experto
   del área).
4. Migrar las coordenadas de cantones a la fuente oficial INEC/IGM (hoy vienen de un gist
   comunitario, con 2 correcciones manuales documentadas).


## 15. Implementación real desplegada

Este sistema no es solo un ejercicio académico: está **desplegado y funcionando** en
producción, de punta a punta (test vocacional -> perfil -> resultados con "Mapa de
afinidad"):

### 🔗 https://tucarrera-ecuador.vercel.app/

- **Frontend:** Vercel (HTML/CSS/JS vanilla).
- **Backend:** Render, API FastAPI (`https://tucarreraecuador.onrender.com`), corriendo el
  mismo motor de recomendación explicado en este notebook, sobre el dataset completo de
  8014 carreras.
- **Test vocacional:** los 60 ítems reales del O*NET Interest Profiler (sección 4).

---
*Recomendador de Carreras Ecuador — Arturo Rodríguez, PhD ([ORCID 0000-0002-7017-9443](https://orcid.org/0000-0002-7017-9443)).
Fuente de datos: SENESCYT, Portal Único de Datos Abiertos del Ecuador. Instrumento vocacional:
National Center for O*NET Development (2010), U.S. Department of Labor.*
